# Notebook 03 — Forecasting Probabilístico con Prophet
**Proyecto:** AgroMétrica · Equipo IA  
**Objetivo:** Predecir ET₀ y precipitación a 14 días generando tres escenarios probabilísticos  
(P25 pesimista · P50 neutro · P75 optimista) para alimentar el motor de déficit hídrico.

**Fuente de datos:** `datos_diarios_parcela.csv` — pipeline Open-Meteo del equipo Data  
**Periodo histórico:** 2005-01-01 → hoy (ERA5 reanalysis)  
**Modelo:** Prophet (regresión aditiva) — `y(t) = g(t) + s(t) + h(t) + ε`

---

## 1. Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics

# Reproducibilidad
np.random.seed(42)

# Configuración de gráficos
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('Librerías cargadas correctamente')

## 2. Carga y validación de datos

In [ ]:
# Carga del CSV generado por el equipo de Data
df = pd.read_csv('../data-ingestion/datos_diarios_parcela.csv', parse_dates=['fecha'])

print(f'Total registros: {len(df):,}')
print(f'Periodo: {df["fecha"].min().date()} → {df["fecha"].max().date()}')
print(f'Fuentes: {df["fuente"].value_counts().to_dict()}')
print(f'Nulos en et0_mm: {df["et0_mm"].isna().sum()}')
print(f'Nulos en precipitacion_mm: {df["precipitacion_mm"].isna().sum()}')

df.head(3)

## 3. Preparación del dataset de entrenamiento

Usamos **únicamente datos históricos ERA5** para entrenar Prophet.  
Los datos `observado_modelo` y `forecast` quedan fuera del entrenamiento para evitar overfitting.

In [ ]:
# Prophet requiere columnas 'ds' (fecha) e 'y' (variable objetivo)
historico = df[df['fuente'] == 'historico_era5'].copy()

# Dataset para ET₀
df_et0 = historico[['fecha', 'et0_mm']].rename(columns={'fecha': 'ds', 'et0_mm': 'y'})

# Dataset para precipitación
df_precip = historico[['fecha', 'precipitacion_mm']].rename(columns={'fecha': 'ds', 'precipitacion_mm': 'y'})

print(f'Registros de entrenamiento: {len(df_et0):,} días ({df_et0["ds"].min().date()} → {df_et0["ds"].max().date()})')
print(f'ET₀ media: {df_et0["y"].mean():.2f} mm/día | std: {df_et0["y"].std():.2f}')
print(f'Precipitación media: {df_precip["y"].mean():.2f} mm/día | std: {df_precip["y"].std():.2f}')

## 4. Análisis exploratorio
Verificamos la estacionalidad anual antes de entrenar. Prophet la detectará automáticamente,  
pero es buena práctica confirmarla visualmente.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# ET₀ últimos 3 años
et0_reciente = df_et0[df_et0['ds'].dt.year >= 2022]
axes[0].plot(et0_reciente['ds'], et0_reciente['y'], color='#e67e22', linewidth=0.8, alpha=0.7)
axes[0].set_title('ET₀ diaria (mm) — últimos 3 años', fontweight='bold')
axes[0].set_ylabel('ET₀ (mm/día)')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

# Precipitación últimos 3 años
precip_reciente = df_precip[df_precip['ds'].dt.year >= 2022]
axes[1].bar(precip_reciente['ds'], precip_reciente['y'], color='#2980b9', alpha=0.7, width=1)
axes[1].set_title('Precipitación diaria (mm) — últimos 3 años', fontweight='bold')
axes[1].set_ylabel('Precipitación (mm/día)')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

plt.tight_layout()
plt.savefig('eda_series_temporales.png', dpi=150, bbox_inches='tight')
plt.show()
print('Estacionalidad anual confirmada: ET₀ alta en verano, baja en invierno.')

## 5. Modelo Prophet — ET₀

Configuración del modelo:
- `seasonality_mode='multiplicative'`: la amplitud de la estacionalidad escala con la tendencia
- `yearly_seasonality=True`: captura el ciclo anual del sol
- `weekly_seasonality=False`: ET₀ no tiene patrón semanal (variable climática, no humana)
- `interval_width=0.5`: los intervalos de incertidumbre corresponden a P25–P75
- `changepoint_prior_scale=0.05`: regularización para evitar overfitting en la tendencia

In [ ]:
modelo_et0 = Prophet(
    seasonality_mode='multiplicative',
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    interval_width=0.5,          # P25 = yhat_lower, P75 = yhat_upper
    changepoint_prior_scale=0.05  # Regularización anti-overfitting
)

modelo_et0.fit(df_et0)
print('Modelo ET₀ entrenado con', len(df_et0), 'observaciones')

### 5.1 Forecast ET₀ — 14 días

In [ ]:
DIAS_FORECAST = 14

futuro_et0 = modelo_et0.make_future_dataframe(periods=DIAS_FORECAST, freq='D')
forecast_et0 = modelo_et0.predict(futuro_et0)

# Extraer solo los 14 días futuros
pred_et0 = forecast_et0[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(DIAS_FORECAST).copy()
pred_et0.columns = ['fecha', 'et0_p50', 'et0_p25', 'et0_p75']

# ET₀ no puede ser negativa
for col in ['et0_p25', 'et0_p50', 'et0_p75']:
    pred_et0[col] = pred_et0[col].clip(lower=0).round(2)

print('Previsión ET₀ — próximos 14 días:')
pred_et0[['fecha', 'et0_p25', 'et0_p50', 'et0_p75']]

## 6. Modelo Prophet — Precipitación

La precipitación tiene una distribución muy asimétrica (muchos días con 0 mm).  
Aplicamos `log1p` antes de entrenar para normalizar la distribución y mejorar las predicciones.

In [ ]:
# Transformación log1p para estabilizar la varianza
df_precip_log = df_precip.copy()
df_precip_log['y'] = np.log1p(df_precip_log['y'])

modelo_precip = Prophet(
    seasonality_mode='additive',
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    interval_width=0.5,
    changepoint_prior_scale=0.05
)

modelo_precip.fit(df_precip_log)
print('Modelo Precipitación entrenado con', len(df_precip_log), 'observaciones')

In [ ]:
futuro_precip = modelo_precip.make_future_dataframe(periods=DIAS_FORECAST, freq='D')
forecast_precip = modelo_precip.predict(futuro_precip)

pred_precip = forecast_precip[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(DIAS_FORECAST).copy()
pred_precip.columns = ['fecha', 'precip_p50', 'precip_p25', 'precip_p75']

# Invertir log1p y clamp a 0
for col in ['precip_p25', 'precip_p50', 'precip_p75']:
    pred_precip[col] = np.expm1(pred_precip[col]).clip(lower=0).round(2)

print('Previsión Precipitación — próximos 14 días:')
pred_precip[['fecha', 'precip_p25', 'precip_p50', 'precip_p75']]

## 7. Construcción de escenarios de déficit hídrico

Combinamos los percentiles de ET₀ y precipitación para generar tres escenarios agrícolas:

| Escenario | ET₀ | Precipitación | Lógica |
|-----------|-----|--------------|--------|
| **Pesimista (P25)** | P75 (alta) | P25 (baja) | Mucho calor, poca lluvia → máximo riego |
| **Neutro (P50)** | P50 | P50 | Condiciones medias esperadas |

In [ ]:
KC = 0.70  # Kc cítricos fase media (FAO-56)

escenarios = pred_et0[['fecha']].copy()

# Escenario PESIMISTA: máxima ET₀, mínima lluvia
escenarios['etc_pesimista']    = (pred_et0['et0_p75'] * KC).round(2)
escenarios['deficit_pesimista'] = (escenarios['etc_pesimista'] - pred_precip['precip_p25']).clip(lower=0).round(2)

# Escenario NEUTRO: valores medianos
escenarios['etc_neutro']       = (pred_et0['et0_p50'] * KC).round(2)
escenarios['deficit_neutro']   = (escenarios['etc_neutro'] - pred_precip['precip_p50']).clip(lower=0).round(2)

# Escenario OPTIMISTA: mínima ET₀, máxima lluvia
escenarios['etc_optimista']    = (pred_et0['et0_p25'] * KC).round(2)
escenarios['deficit_optimista'] = (escenarios['etc_optimista'] - pred_precip['precip_p75']).clip(lower=0).round(2)

print('Déficit hídrico por escenario (mm/día):')
escenarios.set_index('fecha')[['deficit_pesimista', 'deficit_neutro', 'deficit_optimista']]

## 8. Visualización de escenarios

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

fechas = escenarios['fecha']

# ── Gráfico 1: ET₀ con banda de incertidumbre ──
axes[0].fill_between(fechas, pred_et0['et0_p25'], pred_et0['et0_p75'],
                     alpha=0.25, color='#e67e22', label='Banda P25–P75')
axes[0].plot(fechas, pred_et0['et0_p50'], color='#e67e22',
             linewidth=2, marker='o', markersize=4, label='ET₀ P50 (neutro)')
axes[0].set_title('Previsión ET₀ — 14 días (cítricos Valencia)', fontweight='bold', fontsize=12)
axes[0].set_ylabel('ET₀ (mm/día)')
axes[0].legend()
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))

# ── Gráfico 2: Déficit hídrico por escenario ──
axes[1].fill_between(fechas,
                     escenarios['deficit_optimista'],
                     escenarios['deficit_pesimista'],
                     alpha=0.2, color='#e74c3c', label='Rango P25–P75')
axes[1].plot(fechas, escenarios['deficit_pesimista'], color='#c0392b',
             linewidth=2, linestyle='--', label='Pesimista (máx. riego)')
axes[1].plot(fechas, escenarios['deficit_neutro'], color='#e74c3c',
             linewidth=2.5, marker='o', markersize=4, label='Neutro')
axes[1].plot(fechas, escenarios['deficit_optimista'], color='#2980b9',
             linewidth=2, linestyle='--', label='Optimista (mín. riego)')
axes[1].axhline(y=0, color='#27ae60', linewidth=1.5, linestyle='-', alpha=0.7, label='Sin déficit')
axes[1].set_title(f'Déficit hídrico por escenario — Kc={KC} (cítricos)', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Déficit (mm/día)')
axes[1].set_xlabel('Fecha')
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))

plt.tight_layout()
plt.savefig('escenarios_deficit_hidrico.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Validación del modelo — Cross-validation

Prophet incluye su propio framework de validación temporal.  
Entrenamos con los primeros años y medimos el error en los años siguientes,  
sin filtrar información del futuro al pasado (walk-forward validation).

In [ ]:
# Cross-validation: entrenamos con mínimo 5 años, evaluamos en ventanas de 30 días
print('Ejecutando cross-validation (puede tardar ~1 minuto)...')

cv_et0 = cross_validation(
    modelo_et0,
    initial='1825 days',  # 5 años de entrenamiento mínimo
    period='90 days',     # nueva ventana cada 90 días
    horizon='14 days',    # predicción a 14 días
    parallel='processes'
)

metricas_et0 = performance_metrics(cv_et0)
print('\nMétricas ET₀ (horizonte de 14 días):')
print(f'  MAE:  {metricas_et0["mae"].mean():.3f} mm/día')
print(f'  RMSE: {metricas_et0["rmse"].mean():.3f} mm/día')
print(f'  MAPE: {metricas_et0["mape"].mean():.1%}')

## 10. Output — Contrato de datos para model-serving

Generamos el JSON con el formato exacto que consume el endpoint `/recomendar` del equipo de IA.

In [ ]:
import json
from datetime import datetime

def generar_payload_forecast(parcela_id: str, codigo_ine: str, cultivo: str, fase: str, kc: float) -> dict:
    """
    Genera el payload JSON con escenarios probabilísticos P25/P50/P75
    listo para enviar al endpoint /recomendar del model-serving.
    """
    escenarios_list = []
    for _, row in escenarios.iterrows():
        escenarios_list.append({
            'fecha':              row['fecha'].strftime('%Y-%m-%d'),
            'deficit_pesimista':  row['deficit_pesimista'],
            'deficit_neutro':     row['deficit_neutro'],
            'deficit_optimista':  row['deficit_optimista'],
            'etc_pesimista':      row['etc_pesimista'],
            'etc_neutro':         row['etc_neutro'],
            'etc_optimista':      row['etc_optimista'],
        })

    payload = {
        'parcela_id':    parcela_id,
        'codigo_ine':    codigo_ine,
        'cultivo':       cultivo,
        'fase':          fase,
        'kc_aplicado':   kc,
        'generado_en':   datetime.now().isoformat(),
        'dias_forecast': DIAS_FORECAST,
        'escenarios':    escenarios_list
    }
    return payload


# Ejemplo de uso
payload = generar_payload_forecast(
    parcela_id='valencia_citricos',
    codigo_ine='46017',
    cultivo='citricos',
    fase='mediados',
    kc=KC
)

# Guardar
with open('forecast_escenarios.json', 'w') as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print('forecast_escenarios.json generado correctamente')
print(json.dumps(payload['escenarios'][0], indent=2))

## 11. Resumen

| | Valor |
|---|---|
| Datos de entrenamiento | 7.670 días (ERA5, 2005–2025) |
| Modelo | Prophet — regresión aditiva con estacionalidad anual |
| Variables modeladas | ET₀ (mm/día) y Precipitación (mm/día) |
| Horizonte de predicción | 14 días |
| Escenarios | P25 pesimista · P50 neutro · P75 optimista |
| Overfitting control | `changepoint_prior_scale=0.05` + walk-forward CV |

**Decisión de diseño:** los escenarios no son combinaciones aleatorias de percentiles.  
El pesimista usa ET₀ alta (P75) con precipitación baja (P25) porque agronómicamente  
son las condiciones que maximizan el déficit. El optimista invierte la lógica.

**Siguiente paso:** integrar este forecast en el endpoint `/recomendar` del model-serving.